## Génération du Dataset Synthétique 'Heart Disease Prediction'

Pour répondre aux exigences d'utiliser un jeu de données de prédiction des maladies cardiaques, et étant donné les contraintes d'environnement, nous allons générer un dataset synthétique avec des caractéristiques typiques de ce domaine. Ce dataset sera utilisé de manière cohérente dans tous les exercices suivants.

In [67]:
import numpy as np
import pandas as pd

# Fixer la graine pour la reproductibilité
np.random.seed(42)

# Nombre d'échantillons
n_samples = 5000

# Génération des caractéristiques synthétiques
data = {
    'age': np.random.randint(29, 77, n_samples),  # Âge
    'sex': np.random.randint(0, 2, n_samples),   # Sexe (0=femme, 1=homme)
    'cp': np.random.randint(0, 4, n_samples),    # Type de douleur thoracique (0-3)
    'trestbps': np.random.randint(90, 200, n_samples), # Pression artérielle au repos
    'chol': np.random.randint(120, 565, n_samples), # Cholestérol
    'fbs': np.random.randint(0, 2, n_samples),   # Glycémie à jeun > 120 mg/dl (0=faux, 1=vrai)
    'restecg': np.random.randint(0, 3, n_samples), # Résultats d'électrocardiographie au repos (0-2)
    'thalach': np.random.randint(70, 202, n_samples), # Fréquence cardiaque maximale atteinte
    'exang': np.random.randint(0, 2, n_samples),   # Angine de poitrine induite par l'exercice (0=non, 1=oui)
    'oldpeak': np.round(np.random.uniform(0.0, 6.2, n_samples), 1), # Dépression du segment ST induite par l'exercice
    'slope': np.random.randint(0, 3, n_samples),  # Pente du segment ST
    'ca': np.random.randint(0, 5, n_samples),     # Nombre de vaisseaux majeurs colorés par fluoroscopie (0-4)
    'thal': np.random.randint(0, 4, n_samples),  # Thalassémie (0=null, 1=normal, 2=défaut fixe, 3=défaut réversible)
}

# Génération de la variable cible 'HeartDisease'
# Créons une corrélation légère avec certaines caractéristiques pour la vraisemblance
# HeartDisease = 1 si au moins 3 facteurs de risque sont présents ou 2 facteurs très importants.
# Facteurs de risque arbitraires: age, sex=1, cp>0, trestbps>140, chol>240, exang=1, oldpeak>1.0, thal>1

df_heart = pd.DataFrame(data)

df_heart['HeartDisease'] = (
    (df_heart['age'] > 50).astype(int) +
    (df_heart['sex'] == 1).astype(int) +
    (df_heart['cp'] > 0).astype(int) +
    (df_heart['trestbps'] > 130).astype(int) +
    (df_heart['chol'] > 200).astype(int) +
    (df_heart['exang'] == 1).astype(int) +
    (df_heart['oldpeak'] > 1.0).astype(int) +
    (df_heart['thal'] > 1).astype(int)
).apply(lambda x: 1 if x >= 4 else 0)

# Ajout d'un peu de bruit pour rendre la classification non triviale
df_heart['HeartDisease'] = df_heart.apply(lambda row:
    1 - row['HeartDisease'] if np.random.rand() < 0.1 else row['HeartDisease'],
    axis=1
)

print(f"Dataset 'Heart Disease Prediction' généré : {df_heart.shape[0]} lignes, {df_heart.shape[1]} colonnes")
print("Distribution de la cible 'HeartDisease' :")
print(df_heart['HeartDisease'].value_counts(normalize=True))

# Afficher les premières lignes du dataset
display(df_heart.head())


Dataset 'Heart Disease Prediction' généré : 5000 lignes, 14 colonnes
Distribution de la cible 'HeartDisease' :
HeartDisease
1.0    0.8124
0.0    0.1876
Name: proportion, dtype: float64


,age,sex,cp,trestbps,chol,fbs,restecg,thalach,exang,oldpeak,slope,ca,thal,HeartDisease
0,67,1,3,170,291,1,1,164,0,4.2,0,2,3,1.0
1,57,1,2,103,487,1,1,148,0,5.7,2,1,0,1.0
2,43,0,2,160,366,0,0,197,0,4.0,2,4,1,1.0
3,71,1,1,175,366,1,0,147,1,5.2,2,4,3,0.0
4,36,1,3,140,530,0,0,105,1,3.4,0,2,3,1.0


Exercice 1 : Analyse exploratoire des données

Instructions

Charger les données à partir des fichiers CSV
Supprimer la colonne cible des données d'entraînement
Séparer les données en groupes train/test
Comprendre les données

In [68]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

# ====================== CHARGEMENT ======================
# Le dataset 'df_heart' a été généré précédemment
df = df_heart.copy()

print(f"Dataset 'Heart Disease Prediction' chargé : {df.shape[0]} lignes, {df.shape[1]} colonnes")

print("Distribution de la cible 'HeartDisease' :")
print(df['HeartDisease'].value_counts(normalize=True))

Dataset 'Heart Disease Prediction' chargé : 5000 lignes, 14 colonnes
Distribution de la cible 'HeartDisease' :
HeartDisease
1.0    0.8124
0.0    0.1876
Name: proportion, dtype: float64


In [69]:
# Colonne cible
target = 'HeartDisease'

# Données pour l'entraînement (sans la cible)
X = df.drop(columns=[target])
y = df[target]

print(f"Features (X) : {X.shape}")
print(f"Cible (y)   : {y.shape}")

Features (X) : (5000, 13)
Cible (y)   : (5000,)


In [70]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.25,
    random_state=42
)

print(f"Train set : {X_train.shape[0]:,} lignes")
print(f"Test set  : {X_test.shape[0]:,} lignes")
print(f"Ratio train/test : {len(X_train)/len(X):.1%} / {len(X_test)/len(X):.1%}")

Train set : 3,750 lignes
Test set  : 1,250 lignes
Ratio train/test : 75.0% / 25.0%


In [71]:
# Informations générales
print("\n=== INFOS GÉNÉRALES ===")
print(df.info())

print("\n=== STATISTIQUES DESCRIPTIVES ===")
print(df.describe().round(2))

print("\n=== VALEURS MANQUANTES ===")
print(df.isnull().sum().sort_values(ascending=False))

print("\n=== DOUBLONS ===")
print(f"Nombre de doublons : {df.duplicated().sum()}")

# Analyse univariée des colonnes catégorielles spécifiques à df_heart
print("\n=== DISTRIBUTION DES CARACTÉRISTIQUES CATÉGORIELLES ===")
cat_cols = ['sex', 'cp', 'fbs', 'restecg', 'exang', 'slope', 'ca', 'thal']
for col in cat_cols:
    if col in df.columns:
        print(f"\n{col}:")
        print(df[col].value_counts().head(8))


=== INFOS GÉNÉRALES ===
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5000 entries, 0 to 4999
Data columns (total 14 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   age           5000 non-null   int64  
 1   sex           5000 non-null   int64  
 2   cp            5000 non-null   int64  
 3   trestbps      5000 non-null   int64  
 4   chol          5000 non-null   int64  
 5   fbs           5000 non-null   int64  
 6   restecg       5000 non-null   int64  
 7   thalach       5000 non-null   int64  
 8   exang         5000 non-null   int64  
 9   oldpeak       5000 non-null   float64
 10  slope         5000 non-null   int64  
 11  ca            5000 non-null   int64  
 12  thal          5000 non-null   int64  
 13  HeartDisease  5000 non-null   float64
dtypes: float64(2), int64(12)
memory usage: 547.0 KB
None

=== STATISTIQUES DESCRIPTIVES ===
           age     sex       cp  trestbps     chol     fbs  restecg  thalach  \
count  

Exercice 2 : Régression logistique sans recherche par grille

Instructions

Utilisez l'ensemble de données pour construire un modèle de régression logistique sans recourir à une recherche exhaustive. Divisez les données en ensembles d'entraînement et de test, puis entraînez le modèle de régression logistique et évaluez ses performances sur l'ensemble de test.



In [94]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, roc_auc_score, confusion_matrix, classification_report)

# ========================== 1. CHARGEMENT ==========================
# Le dataset 'df_heart' a été généré précédemment
df = df_heart.copy()

print(f"Dataset 'Heart Disease Prediction' chargé : {df.shape[0]} lignes, {df.shape[1]} colonnes")

print("Distribution de la cible 'HeartDisease' :")
print(df['HeartDisease'].value_counts(normalize=True))

Dataset 'Heart Disease Prediction' chargé : 5000 lignes, 14 colonnes
Distribution de la cible 'HeartDisease' :
HeartDisease
1.0    0.8124
0.0    0.1876
Name: proportion, dtype: float64


In [87]:
# Features
# Using the feature list derived from df_heart
features = ['age', 'sex', 'cp', 'trestbps', 'chol', 'fbs', 'restecg', 'thalach', 'exang', 'oldpeak', 'slope', 'ca', 'thal']

X = df[features]
y = df['HeartDisease']

print(f"\nFeatures utilisées : {features}")


Features utilisées : ['age', 'sex', 'cp', 'trestbps', 'chol', 'fbs', 'restecg', 'thalach', 'exang', 'oldpeak', 'slope', 'ca', 'thal']


In [88]:
# Séparation Train / Test
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)

print(f"Train set : {X_train.shape[0]:,} lignes")
print(f"Test set  : {X_test.shape[0]:,} lignes")

# Pipeline de prétraitement
categorical_features = ['sex', 'cp', 'fbs', 'restecg', 'exang', 'slope', 'ca', 'thal']
numerical_features = ['age', 'trestbps', 'chol', 'thalach', 'oldpeak']

preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numerical_features),
        ('cat', OneHotEncoder(drop='first', handle_unknown='ignore'), categorical_features)
    ])

Train set : 3,750 lignes
Test set  : 1,250 lignes


In [92]:
# Modèle sans GridSearch (hyperparamètres par défaut + max_iter)
model = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', LogisticRegression(
        random_state=42,
        max_iter=1000,
        solver='lbfgs'
    ))
])

# Entraînement
model.fit(X_train, y_train)
print("\n Modèle de Régression Logistique entraîné")


 Modèle de Régression Logistique entraîné


In [93]:
# Prédictions
y_pred = model.predict(X_test)
y_pred_proba = model.predict_proba(X_test)[:, 1]

# Métriques
print("\n" + "="*60)
print("RÉSULTATS DE LA RÉGRESSION LOGISTIQUE")
print("="*60)
print(f"Accuracy          : {accuracy_score(y_test, y_pred):.4f}")
print(f"Precision         : {precision_score(y_test, y_pred):.4f}")
print(f"Recall            : {recall_score(y_test, y_pred):.4f}")
print(f"F1-Score          : {f1_score(y_test, y_pred):.4f}")
print(f"ROC-AUC           : {roc_auc_score(y_test, y_pred_proba):.4f}")

print("\nRapport de classification :")
# Update target_names for Heart Disease
print(classification_report(y_test, y_pred, target_names=['No Heart Disease', 'Heart Disease']))

print("\nMatrice de confusion :")
print(confusion_matrix(y_test, y_pred))


RÉSULTATS DE LA RÉGRESSION LOGISTIQUE
Accuracy          : 0.8312
Precision         : 0.8356
Recall            : 0.9862
F1-Score          : 0.9047
ROC-AUC           : 0.7384

Rapport de classification :
                  precision    recall  f1-score   support

No Heart Disease       0.73      0.16      0.26       235
   Heart Disease       0.84      0.99      0.90      1015

        accuracy                           0.83      1250
       macro avg       0.78      0.57      0.58      1250
    weighted avg       0.82      0.83      0.78      1250


Matrice de confusion :
[[  38  197]
 [  14 1001]]


Exercice 3 : Régression logistique avec recherche par grille

Instructions

Construisez un modèle de régression logistique à l'aide de l'ensemble de données, mais cette fois-ci, utilisez GridSearchCV pour optimiser les hyperparamètres tels que C et la pénalité.



In [125]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, classification_report, confusion_matrix

# ========================== 1. CHARGEMENT ==========================
# Le dataset 'df_heart' a été généré précédemment dans l'exercice 1
df = df_heart.copy()

print(f"Dataset 'Heart Disease Prediction' chargé : {df.shape[0]} lignes, {df.shape[1]} colonnes")

# ========================== 2. PRÉPARATION DES DONNÉES ==========================
# Colonne cible
target = 'HeartDisease'

# Features et cible
X = df.drop(columns=[target])
y = df[target]

# Séparation Train / Test
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)

print(f"\nTrain set : {X_train.shape[0]:,} lignes")
print(f"Test set  : {X_test.shape[0]:,} lignes")

# Pipeline de prétraitement (identique à l'Exercice 2)
categorical_features = ['sex', 'cp', 'fbs', 'restecg', 'exang', 'slope', 'ca', 'thal']
numerical_features = ['age', 'trestbps', 'chol', 'thalach', 'oldpeak']

preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numerical_features),
        ('cat', OneHotEncoder(drop='first', handle_unknown='ignore'), categorical_features)
    ])

pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', LogisticRegression(
        random_state=42,
        max_iter=1000 # Augmenter pour la convergence
    ))
])

# Grille d'hyperparamètres pour LogisticRegression
param_grid = {
    'classifier__C': [0.01, 0.1, 1, 10, 100],
    'classifier__penalty': ['l1', 'l2'],
    'classifier__solver': ['liblinear', 'lbfgs'] # liblinear supporte l1 et l2, lbfgs supporte l2
}

# GridSearchCV
grid_search = GridSearchCV(
    pipeline, param_grid, cv=5, scoring='f1', n_jobs=-1, verbose=0
)

# Entraînement
grid_search.fit(X_train, y_train)
print("\nGridSearchCV pour LogisticRegression terminé")

Dataset 'Heart Disease Prediction' chargé : 5000 lignes, 14 colonnes

Train set : 3,750 lignes
Test set  : 1,250 lignes

GridSearchCV pour LogisticRegression terminé


In [18]:
features = ['Ship_Mode', 'Segment', 'Region', 'Category', 'Sub_Category',
            'Sales', 'Quantity', 'Discount']

X = df[features]
y = df['Profitable']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)

In [78]:
# Préprocesseur
preprocessor = ColumnTransformer([
    ('num', StandardScaler(), ['Sales', 'Quantity', 'Discount']),
    ('cat', OneHotEncoder(drop='first', handle_unknown='ignore'),
     ['Ship_Mode', 'Segment', 'Region', 'Category', 'Sub_Category'])
])

pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', LogisticRegression(random_state=42, max_iter=1000))
])

# Grille d'hyperparamètres
param_grid = {
    'classifier__C': [0.01, 0.1, 1, 10, 100],
    'classifier__penalty': ['l1', 'l2'],
    'classifier__solver': ['liblinear', 'lbfgs']
}

grid_search = GridSearchCV(
    pipeline, param_grid, cv=5, scoring='f1', n_jobs=-1
)

grid_search.fit(X_train, y_train)

GridSearchCV(cv=5,
             estimator=Pipeline(steps=[('preprocessor',
                                        ColumnTransformer(transformers=[('num',
                                                                         StandardScaler(),
                                                                         ['Sales',
                                                                          'Quantity',
                                                                          'Discount']),
                                                                        ('cat',
                                                                         OneHotEncoder(drop='first',
                                                                                       handle_unknown='ignore'),
                                                                         ['Ship_Mode',
                                                                          'Segment',
                                                                          'Region',
                                                                          'Category',
                                                                          'Sub_Category'])])),
                                       ('classifier',
                                        LogisticRegression(max_iter=1000,
                                                           random_state=42))]),
             n_jobs=-1,
             param_grid={'classifier__C': [0.01, 0.1, 1, 10, 100],
                         'classifier__penalty': ['l1', 'l2'],
                         'classifier__solver': ['liblinear', 'lbfgs']},
             scoring='f1')

In [134]:
import numpy as np
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, accuracy_score

# Data
X, y = make_classification(n_samples=1000, n_features=10, random_state=42)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Scale
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Model and grid
log_reg = LogisticRegression(solver='liblinear', max_iter=1000, random_state=42)
param_grid = {'C': [0.01, 0.1, 1, 10, 100], 'penalty': ['l1', 'l2']}

# Grid search
grid_search = GridSearchCV(log_reg, param_grid, cv=5, scoring='accuracy', n_jobs=-1, verbose=1)
grid_search.fit(X_train_scaled, y_train)

# Results
print("Best params:", grid_search.best_params_)
print("Best CV accuracy: {:.3f}".format(grid_search.best_score_))

best_model = grid_search.best_estimator_
y_pred = best_model.predict(X_test_scaled)
print("Test accuracy: {:.3f}".format(accuracy_score(y_test, y_pred)))
print(classification_report(y_test, y_pred))

Fitting 5 folds for each of 10 candidates, totalling 50 fits
Best params: {'C': 0.01, 'penalty': 'l2'}
Best CV accuracy: 0.876
Test accuracy: 0.830
              precision    recall  f1-score   support

           0       0.77      0.89      0.82        89
           1       0.90      0.78      0.84       111

    accuracy                           0.83       200
   macro avg       0.83      0.84      0.83       200
weighted avg       0.84      0.83      0.83       200



Exercice 4 : SVM sans recherche par grille

Instructions

Entraînez un classificateur SVM (Support Vector Machine) sur l'ensemble de données sans utiliser de recherche par grille. Choisissez un noyau approprié et définissez manuellement les hyperparamètres.





In [133]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, roc_auc_score, classification_report, confusion_matrix)
import warnings
warnings.filterwarnings("ignore")

# 1. Chargement
# Le dataset 'df_heart' a été généré précédemment
df = df_heart.copy()

print(f"Dataset 'Heart Disease Prediction' chargé : {df.shape[0]} lignes, {df.shape[1]} colonnes")

# 2. Préparation des données
target = 'HeartDisease'

X = df.drop(columns=[target])
y = df[target]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)

print(f"\nTrain set : {X_train.shape[0]:,} lignes")
print(f"Test set  : {X_test.shape[0]:,} lignes")

# 3. Pipeline de prétraitement (identique à l'Exercice 2)
categorical_features = ['sex', 'cp', 'fbs', 'restecg', 'exang', 'slope', 'ca', 'thal']
numerical_features = ['age', 'trestbps', 'chol', 'thalach', 'oldpeak']

preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numerical_features),
        ('cat', OneHotEncoder(drop='first', handle_unknown='ignore'), categorical_features)
    ])

# 4. SVM avec hyperparamètres choisis manuellement
svm_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', SVC(
        kernel='rbf',      # Noyau RBF (Radial Basis Function)
        C=1.0,             # Paramètre de régularisation
        gamma='scale',     # Auto scaling basé sur le nombre de features
        probability=True,  # Nécessaire pour roc_auc_score
        random_state=42,
        max_iter=5000      # Augmenter pour la convergence
    ))
])

# Entraînement
svm_pipeline.fit(X_train, y_train)
print("\nModèle SVM entraîné")

Dataset 'Heart Disease Prediction' chargé : 5000 lignes, 14 colonnes

Train set : 3,750 lignes
Test set  : 1,250 lignes

Modèle SVM entraîné


In [128]:
# ============================================================
# Exercice 4 : SVM SANS RECHERCHE PAR GRILLE
# ============================================================

import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.decomposition import PCA
from sklearn.model_selection import cross_val_score
import os

# ─────────────────────────────────────────────
# 1. PRÉTRAITEMENT & ENTRAÎNEMENT
# ─────────────────────────────────────────────
print("=" * 60)
print("   EXERCICE 4 : SVM SANS RECHERCHE PAR GRILLE")
print("=" * 60)

print("\n" + "─" * 60)
print("  ÉTAPE 1 : Justification du Prétraitement et Hyperparamètres")
print("─" * 60)

print("   • Dataset : Heart Disease Prediction (déjà chargé et splité)")
print("   • Prétraitement : StandardScaler pour les numériques, OneHotEncoder pour les catégorielles (via Pipeline).")
print("     → Nécessaire car le SVM est sensible à l'échelle des données.")

print("\n   Noyaux disponibles et leur usage :\n   ┌─────────────┬────────────────────────────────────────────┐\n   │ Noyau       │ Cas d'usage                                │\n   ├─────────────┼────────────────────────────────────────────┤\n   │ linear      │ Données séparables linéairement            │\n   │ poly        │ Relations polynomiales entre features      │\n   │ rbf (RBF)   │ Cas général, frontières non-linééraires  ★  │\n   │ sigmoid     │ Inspiré des réseaux de neurones            │\n   └─────────────┴────────────────────────────────────────────┘")
print("\n   → Choix retenu : kernel='rbf' (Radial Basis Function)")
print("     Justification :")
print("     • Le dataset de prédiction de maladies cardiaques peut présenter des relations complexes et non-linéaires.")
print("     • RBF est flexible et performant même sans connaissance a priori de la structure sous-jacente des données.")
print("     • C'est un choix standard et robuste pour de nombreux problèmes de classification.")

print("\n   Hyperparamètres du SVM (RBF) :\n")
print("   • C = 1.0  (régularisation)")
print("     - C faible  → marge large, plus de violations tolérées (underfitting)")
print("     - C élevé   → marge étroite, peu de violations (risque overfitting)")
print("     → C=1.0 : valeur équilibrée, bon compromis biais/variance.")

print("   • gamma = 'scale'  (influence d'un seul exemple)")
print("     - gamma faible  → influence large, frontière lisse")
print("     - gamma élevé   → influence locale, frontière complexe")
print("     → 'scale' = 1 / (n_features × Var(X)) : adaptatif et robuste, recommandé comme valeur par défaut.")

# Le modèle est déjà entraîné dans la cellule précédente: `svm_pipeline`
svm_model = svm_pipeline

# ─────────────────────────────────────────────
# 2. ÉVALUATION
# ─────────────────────────────────────────────
print("\n" + "─" * 60)
print("  ÉTAPE 2 : Évaluation du modèle")
print("─" * 60)

y_pred  = svm_model.predict(X_test)
y_proba = svm_model.predict_proba(X_test)[:, 1]

acc   = accuracy_score(y_test, y_pred)
auc   = roc_auc_score(y_test, y_proba)
cv_scores = cross_val_score(svm_model, X_train, y_train, cv=5, scoring='accuracy')

print(f"\n   Accuracy  (test)    : {acc:.4f}  ({acc*100:.2f}%) ")
print(f"   AUC-ROC   (test)    : {auc:.4f}")
print(f"   CV Accuracy (5-fold): {cv_scores.mean():.4f} ± {cv_scores.std():.4f}")

print(f"\n   Rapport de classification :\n")
print(classification_report(y_test, y_pred, target_names=['No Heart Disease', 'Heart Disease']))

# ─────────────────────────────────────────────
# 3. VISUALISATIONS (Adaptées pour df_heart)
# ─────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
fig.suptitle("Exercice 4 – SVM (noyau RBF) sans GridSearchCV\nDataset : Heart Disease Prediction",
             fontsize=13, fontweight='bold', y=1.05)

# (A) Matrice de confusion
ax = axes[0]
cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
            xticklabels=['No Heart Disease', 'Heart Disease'],
            yticklabels=['No Heart Disease', 'Heart Disease'],
            linewidths=1, annot_kws={'size': 14})
ax.set_title("Matrice de Confusion (RBF)", fontweight='bold')
ax.set_ylabel("Vraie classe")
ax.set_xlabel("Classe prédite")

# (B) Courbe ROC
ax = axes[1]
fpr, tpr, _ = roc_curve(y_test, y_proba)
ax.plot(fpr, tpr, 'b-', linewidth=2, label=f'RBF (AUC = {auc:.4f})')
ax.plot([0,1],[0,1],'k--',linewidth=1, label='Aléatoire (AUC = 0.50)')
ax.fill_between(fpr, tpr, alpha=0.07, color='blue')
ax.set_title("Courbe ROC", fontweight='bold')
ax.set_xlabel("Taux de Faux Positifs (FPR)")
ax.set_ylabel("Taux de Vrais Positifs (TPR)")
ax.legend(fontsize=9)
ax.grid(alpha=0.3)

plt.tight_layout()
os.makedirs('/mnt/user-data/outputs/', exist_ok=True)
plt.savefig('/mnt/user-data/outputs/exercice4_svm.png',
            dpi=150, bbox_inches='tight')
plt.close()
print("\n   📊 Visualisations sauvegardées ✓")

# ─────────────────────────────────────────────
# 4. RÉSUMÉ FINAL
# ─────────────────────────────────────────────
print("\n" + "=" * 60)
print("  RÉSUMÉ FINAL")
print("=" * 60)
print(f"   Noyau choisi    : RBF (Radial Basis Function)")
print(f"   C               : 1.0   | Gamma : scale")
print(f"   Accuracy (test) : {acc:.4f} ({acc*100:.2f}%) ")
print(f"   AUC-ROC         : {auc:.4f}")
print(f"   CV 5-fold       : {cv_scores.mean():.4f} ± {cv_scores.std():.4f}")
print("\n✅ Exercice 4 terminé avec succès !\n")

   EXERCICE 4 : SVM SANS RECHERCHE PAR GRILLE

────────────────────────────────────────────────────────────
  ÉTAPE 1 : Justification du Prétraitement et Hyperparamètres
────────────────────────────────────────────────────────────
   • Dataset : Heart Disease Prediction (déjà chargé et splité)
   • Prétraitement : StandardScaler pour les numériques, OneHotEncoder pour les catégorielles (via Pipeline).
     → Nécessaire car le SVM est sensible à l'échelle des données.

   Noyaux disponibles et leur usage :
   ┌─────────────┬────────────────────────────────────────────┐
   │ Noyau       │ Cas d'usage                                │
   ├─────────────┼────────────────────────────────────────────┤
   │ linear      │ Données séparables linéairement            │
   │ poly        │ Relations polynomiales entre features      │
   │ rbf (RBF)   │ Cas général, frontières non-linééraires  ★  │
   │ sigmoid     │ Inspiré des réseaux de neurones            │
   └─────────────┴──────────────────────

Exercice 5 : SVM avec recherche par grille

Instructions

Implémentez un classificateur SVM sur l'ensemble de données avec GridSearchCV pour trouver la meilleure combinaison des hyperparamètres C, noyau et gamma.

In [130]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, roc_auc_score, classification_report, confusion_matrix)
import warnings
warnings.filterwarnings("ignore")

# 1. Chargement & Préparation
# Le dataset 'df_heart' a été généré précédemment
df = df_heart.copy()

print(f"Dataset 'Heart Disease Prediction' chargé : {df.shape[0]} lignes, {df.shape[1]} colonnes")

target = 'HeartDisease'

X = df.drop(columns=[target])
y = df[target]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)

print(f"\nTrain set : {X_train.shape[0]:,} lignes")
print(f"Test set  : {X_test.shape[0]:,} lignes")

# 2. Pipeline de prétraitement (identique à l'Exercice 2)
categorical_features = ['sex', 'cp', 'fbs', 'restecg', 'exang', 'slope', 'ca', 'thal']
numerical_features = ['age', 'trestbps', 'chol', 'thalach', 'oldpeak']

preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numerical_features),
        ('cat', OneHotEncoder(drop='first', handle_unknown='ignore'), categorical_features)
    ])

pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', SVC(probability=True, random_state=42, max_iter=5000)) # Augmenter max_iter
])

# 3. GridSearchCV
param_grid = [
    {
        'classifier__kernel': ['rbf'],
        'classifier__C'     : [0.1, 1, 10],
        'classifier__gamma' : [0.01, 'scale'],
        'classifier__class_weight': [None, 'balanced']
    },
    {
        'classifier__kernel': ['linear'],
        'classifier__C'     : [0.1, 1, 10],
        'classifier__class_weight': [None, 'balanced']
    }
]

grid_search = GridSearchCV(
    pipeline, param_grid, cv=3, scoring='f1_weighted', n_jobs=-1, verbose=0
)

grid_search.fit(X_train, y_train)

print("GridSearchCV pour SVM terminé")
print("Meilleurs hyperparamètres :", grid_search.best_params_)

Dataset 'Heart Disease Prediction' chargé : 5000 lignes, 14 colonnes

Train set : 3,750 lignes
Test set  : 1,250 lignes
GridSearchCV pour SVM terminé
Meilleurs hyperparamètres : {'classifier__C': 10, 'classifier__class_weight': None, 'classifier__gamma': 'scale', 'classifier__kernel': 'rbf'}


In [111]:
# ============================================================
# Exercice 5 : SVM avec GridSearchCV
# ============================================================

import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from sklearn.decomposition import PCA
from sklearn.model_selection import cross_val_score
import os

# ─────────────────────────────────────────────
# 1. MODÈLE DE RÉFÉRENCE (Exercice 4)
# ─────────────────────────────────────────────
print("=" * 62)
print("   EXERCICE 5 : SVM AVEC GRIDSEARCHCV")
print("=" * 62)

print("\n" + "─" * 62)
print("  ÉTAPE 1 : Modèle de référence (Exercice 4)")
print("─" * 62)

# Reprendre le pipeline SVM de l'Exercice 4 pour la comparaison
# Noter: X_train et X_test sont déjà prétraités par le pipeline.
base_svm_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', SVC(
        kernel='rbf', C=1.0, gamma='scale',
        probability=True, random_state=42, max_iter=5000
    ))
])
base_svm_pipeline.fit(X_train, y_train)

base_acc = accuracy_score(y_test, base_svm_pipeline.predict(X_test))
base_auc = roc_auc_score(y_test, base_svm_pipeline.predict_proba(X_test)[:, 1])
print(f"   • Accuracy  : {base_acc:.4f}  |  AUC : {base_auc:.4f}")
print(f"   • Params    : kernel=rbf, C=1.0, gamma=scale")

# ─────────────────────────────────────────────
# 2. RÉSULTATS GRIDSEARCHCV
# ─────────────────────────────────────────────
print("\n" + "─" * 62)
print("  ÉTAPE 2 : Résultats GridSearchCV")
print("─" * 62)

best_params   = grid_search.best_params_
best_cv_score = grid_search.best_score_

print(f"\n   ✅ Meilleurs hyperparamètres :")
for k, v in best_params.items():
    print(f"      • {k:20s} = {v}")
print(f"\n   📈 Meilleur score CV (f1_weighted) : {best_cv_score:.4f} ({best_cv_score*100:.2f}%) ")

results_df = pd.DataFrame(grid_search.cv_results_)
top5 = (results_df[['param_classifier__kernel','param_classifier__C','param_classifier__gamma',
                     'mean_test_score','std_test_score']]
        .sort_values('mean_test_score', ascending=False).head(5))
print(f"\n   🏆 Top 5 combinaisons :")
print(f"   {'Noyau':>8}  {'C':>7}  {'Gamma':>8}  {'CV Mean':>9}  {'Std':>7}")
print("   " + "─" * 47)
for _, r in top5.iterrows():
    gamma_val = r['param_classifier__gamma']
    gamma_str = str(gamma_val) if not pd.isna(gamma_val) else 'nan'
    print(f"   {str(r['param_classifier__kernel']):>8}  {str(r['param_classifier__C']):>7}  "
          f"{gamma_str:>8}  {r['mean_test_score']:>9.4f}  "
          f"{r['std_test_score']:>7.4f}")

# ─────────────────────────────────────────────
# 3. ÉVALUATION DU MEILLEUR MODÈLE
# ─────────────────────────────────────────────
print("\n" + "─" * 62)
print("  ÉTAPE 3 : Évaluation du meilleur modèle")
print("─" * 62)

best_model = grid_search.best_estimator_
y_pred     = best_model.predict(X_test)
y_proba    = best_model.predict_proba(X_test)[:, 1]

best_acc = accuracy_score(y_test, y_pred)
best_auc = roc_auc_score(y_test, y_proba)
gain_acc = (best_acc - base_acc) * 100
gain_auc = (best_auc - base_auc) * 100

print(f"""
   ┌─────────────────────┬──────────────┬──────────────┬──────────┐
   │ Métrique            │ Exercice 4   │ Exercice 5   │   Gain   │
   ├─────────────────────┼──────────────┼──────────────┼──────────┤
   │ Accuracy (test)     │ {base_acc:.4f}       │ {best_acc:.4f}       │ {gain_acc:+.2f}%  │
   │ AUC-ROC             │ {base_auc:.4f}       │ {best_auc:.4f}       │ {gain_auc:+.2f}%  │
   └─────────────────────┴──────────────┴──────────────┴──────────┘
""")

print(f"   Rapport de classification :\n")
print(classification_report(y_test, y_pred, target_names=['No Heart Disease', 'Heart Disease']))

# ─────────────────────────────────────────────
# 4. VISUALISATIONS
# ─────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
fig.suptitle("Exercice 5 – SVM avec GridSearchCV\nDataset : Heart Disease Prediction",
             fontsize=13, fontweight='bold', y=1.05)

# (A) Matrice de confusion
ax = axes[0]
cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
            xticklabels=['No Heart Disease', 'Heart Disease'],
            yticklabels=['No Heart Disease', 'Heart Disease'],
            linewidths=1, annot_kws={'size': 14})
ax.set_title(f"Matrice de Confusion\n(Best: {best_params['classifier__kernel'].upper()},\n C={best_params['classifier__C']}, gamma={best_params.get('classifier__gamma', '—')})",
             fontweight='bold')
ax.set_ylabel("Vraie classe")
ax.set_xlabel("Classe prédite")

# (B) Courbe ROC — base vs best
ax = axes[1]
fpr_b, tpr_b, _ = roc_curve(y_test, base_svm_pipeline.predict_proba(X_test)[:, 1])
fpr_g, tpr_g, _ = roc_curve(y_test, y_proba)
ax.plot(fpr_b, tpr_b, '--', color='gray', linewidth=1.8,
        label=f'Exo4 base (AUC={base_auc:.4f})')
ax.plot(fpr_g, tpr_g, '-',  color='blue', linewidth=2.5,
        label=f'GridSearch (AUC={best_auc:.4f})')
ax.fill_between(fpr_g, tpr_g, alpha=0.12, color='blue')
ax.plot([0, 1], [0, 1], 'k:', linewidth=1)
ax.set_title("Courbe ROC — Avant vs Après GridSearch", fontweight='bold')
ax.set_xlabel("FPR")
ax.set_ylabel("TPR")
ax.legend(fontsize=9)
ax.grid(alpha=0.3)

plt.tight_layout()
os.makedirs('/mnt/user-data/outputs/', exist_ok=True)
plt.savefig('/mnt/user-data/outputs/exercice5_svm_gridsearch.png',
            dpi=150, bbox_inches='tight')
plt.close()
print("   📊 Visualisations sauvegardées ✓")

# ─────────────────────────────────────────────
# 5. RÉSUMÉ FINAL
# ─────────────────────────────────────────────
print("\n" + "=" * 62)
print("  RÉSUMÉ FINAL")
print("=" * 62)
print(f"   Meilleurs hyperparamètres :")
for k, v in best_params.items():
    print(f"     • {k:20s} = {v}")
print(f"\n   Accuracy avant (Exo 4) : {base_acc:.4f}")
print(f"   Accuracy après (Exo 5) : {best_acc:.4f}  ({gain_acc:+.2f}%) ")
print(f"   AUC-ROC                : {best_auc:.4f}  ({gain_auc:+.2f}%) ")
print("\n✅ Exercice 5 terminé avec succès !\n")

   EXERCICE 5 : SVM AVEC GRIDSEARCHCV

──────────────────────────────────────────────────────────────
  ÉTAPE 1 : Modèle de référence (Exercice 4)
──────────────────────────────────────────────────────────────
   • Accuracy  : 0.8392  |  AUC : 0.7437
   • Params    : kernel=rbf, C=1.0, gamma=scale

──────────────────────────────────────────────────────────────
  ÉTAPE 2 : Résultats GridSearchCV
──────────────────────────────────────────────────────────────

   ✅ Meilleurs hyperparamètres :
      • classifier__C        = 10
      • classifier__class_weight = None
      • classifier__gamma    = scale
      • classifier__kernel   = rbf

   📈 Meilleur score CV (f1_weighted) : 0.8087 (80.87%) 

   🏆 Top 5 combinaisons :
      Noyau        C     Gamma    CV Mean      Std
   ───────────────────────────────────────────────
        rbf     10.0     scale     0.8087   0.0082
        rbf      1.0     scale     0.7917   0.0024
        rbf     10.0     scale     0.7834   0.0120
        rbf      1.0

Exercice 6 : XGBoost sans recherche par grille

Instructions

Utilisez l'ensemble de données pour entraîner un classificateur XGBoost sans optimisation des hyperparamètres. Définissez les hyperparamètres manuellement et justifiez vos choix.



In [112]:
import pandas as pd
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, roc_auc_score, classification_report, confusion_matrix)
import warnings
warnings.filterwarnings('ignore')

# 1. Chargement
# Le dataset 'df_heart' a été généré précédemment
df = df_heart.copy()

print(f"Dataset 'Heart Disease Prediction' chargé : {df.shape[0]} lignes, {df.shape[1]} colonnes")

# 2. Préparation des données
target = 'HeartDisease'

X = df.drop(columns=[target])
y = df[target]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)

print(f"\nTrain set : {X_train.shape[0]:,} lignes")
print(f"Test set  : {X_test.shape[0]:,} lignes")

# 3. Pipeline de prétraitement (identique à l'Exercice 2)
categorical_features = ['sex', 'cp', 'fbs', 'restecg', 'exang', 'slope', 'ca', 'thal']
numerical_features = ['age', 'trestbps', 'chol', 'thalach', 'oldpeak']

preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numerical_features),
        ('cat', OneHotEncoder(drop='first', handle_unknown='ignore', sparse_output=False), categorical_features)
    ])

# 4. XGBoost avec hyperparamètres choisis manuellement
# Justification des hyperparamètres:
#   - n_estimators=200: Un nombre raisonnable d'arbres pour capturer les relations sans sur-apprendre.
#   - max_depth=6: Contrôle la complexité des arbres. Une profondeur modérée aide à éviter l'overfitting.
#   - learning_rate=0.1: Taux d'apprentissage standard qui équilibre la vitesse de convergence et la robustesse.
#   - subsample=0.8: Sous-échantillonne les données d'entraînement pour chaque arbre, ce qui réduit la variance.
#   - colsample_bytree=0.8: Sous-échantillonne les features pour chaque arbre, réduisant également la variance.
#   - eval_metric='logloss': Une bonne métrique pour les problèmes de classification binaire.
#   - use_label_encoder=False: Supprime l'avertissement de dépréciation.
#   - scale_pos_weight: Peut être utile pour les classes déséquilibrées. Ici, on l'estime à la volée.

scale_pos_weight_val = (y_train == 0).sum() / (y_train == 1).sum()

xgb_model = xgb.XGBClassifier(
    n_estimators=200,
    max_depth=6,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    eval_metric='logloss',
    use_label_encoder=False,
    scale_pos_weight=scale_pos_weight_val # Gérer le déséquilibre de classes
)

pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', xgb_model)
])

pipeline.fit(X_train, y_train)
print("\nModèle XGBoost entraîné")

Dataset 'Heart Disease Prediction' chargé : 5000 lignes, 14 colonnes

Train set : 3,750 lignes
Test set  : 1,250 lignes

Modèle XGBoost entraîné


Exercice 7 : XGBoost avec recherche par grille

Instructions

Entraînez un classificateur XGBoost sur l'ensemble de données en utilisant GridSearchCV pour optimiser les hyperparamètres tels que learning_rate, n_estimators, max_depth, etc.

In [113]:
import pandas as pd
import xgboost as xgb
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, classification_report, confusion_matrix
import warnings
warnings.filterwarnings('ignore')
import numpy as np

# 1. Chargement & cible
# Le dataset 'df_heart' a été généré précédemment
df = df_heart.copy()

print(f"Dataset 'Heart Disease Prediction' chargé : {df.shape[0]} lignes, {df.shape[1]} colonnes")

target = 'HeartDisease'

X = df.drop(columns=[target])
y = df[target]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)

print(f"\nTrain set : {X_train.shape[0]:,} lignes")
print(f"Test set  : {X_test.shape[0]:,} lignes")

# 2. Pipeline de prétraitement (identique à l'Exercice 2)
categorical_features = ['sex', 'cp', 'fbs', 'restecg', 'exang', 'slope', 'ca', 'thal']
numerical_features = ['age', 'trestbps', 'chol', 'thalach', 'oldpeak']

preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numerical_features),
        ('cat', OneHotEncoder(drop='first', handle_unknown='ignore', sparse_output=False), categorical_features)
    ])

# Calculer scale_pos_weight pour gérer le déséquilibre de classes
scale_pos_weight_val = (y_train == 0).sum() / (y_train == 1).sum()

pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', xgb.XGBClassifier(
        random_state=42,
        eval_metric='logloss',
        use_label_encoder=False,
        scale_pos_weight=scale_pos_weight_val
    ))
])

# 3. GridSearchCV
param_grid = {
    'classifier__n_estimators': [100, 200, 300],
    'classifier__max_depth': [3, 5, 7],
    'classifier__learning_rate': [0.05, 0.1, 0.2],
    'classifier__subsample': [0.7, 0.9] # Ajout de subsample pour plus de régularisation
}

grid_search = GridSearchCV(
    pipeline, param_grid, cv=3, scoring='f1_weighted', n_jobs=-1, verbose=0
)

grid_search.fit(X_train, y_train)
print("\nGridSearchCV pour XGBoost terminé")
print("Meilleurs hyperparamètres :", grid_search.best_params_)

Dataset 'Heart Disease Prediction' chargé : 5000 lignes, 14 colonnes

Train set : 3,750 lignes
Test set  : 1,250 lignes

GridSearchCV pour XGBoost terminé
Meilleurs hyperparamètres : {'classifier__learning_rate': 0.1, 'classifier__max_depth': 7, 'classifier__n_estimators': 300, 'classifier__subsample': 0.9}
